# 02 - Annotation inspection
Notebook 01 found that the real annotations are compressed (`.xz`) and live under `annotations/{original,cropped}/{cars,spots}`. This notebook opens them and prints facts: what files exist, what is inside, and what the dataset README says about the format. Nothing is interpreted or invented.

**Inputs:** the attached MetaPKLot dataset. **Outputs:** `/kaggle/working/artifacts/annotations/probe.json`.

Run all cells, then paste the output of cells 3-6 back.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Step 2: configuration. The dataset folder is detected automatically.
DATASET_HINT = ""       # only needed if several datasets are attached: part of the right path, e.g. "metapklot"
DATA_ROOT = None        # or set the path yourself to skip detection
OUT = "/kaggle/working/artifacts/annotations"

from src.data.discover import describe, find_dataset_root
if DATA_ROOT is None:
    DATA_ROOT = str(find_dataset_root("/kaggle/input", DATASET_HINT))
info = describe(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("files:", info["files"], "| top-level entries:", info["n_children"], info["children"])

In [ ]:
# Cell 3: the dataset's own README and tools (the README should define the annotation format)
from pathlib import Path
root = Path(DATA_ROOT)
print((root / "README.md").read_text(errors="ignore")[:7000])
print()
print("---- tools/ ----")
for p in sorted((root / "tools").rglob("*"))[:60]:
    if p.is_file():
        print(p.relative_to(root), p.stat().st_size)

In [ ]:
# Cell 4: every folder under annotations/ and non_*_days: file counts, sizes, sample names
from src.data.annotation_probe import dir_summary
rows = []
for sub in ["annotations", "non_representative_days", "non_working_days"]:
    if (root / sub).exists():
        for r in dir_summary(root / sub):
            r["dir"] = os.path.relpath(r["dir"], str(root))
            rows.append(r)
            print(r["dir"], "| files:", r["files"], r["extensions"], "| MB:", r["total_mb"], "| e.g.", r["sample"][:3])

In [ ]:
# Cell 5: open one file from every annotation folder and show what is inside
from src.data.annotation_probe import first_file_per_dir, probe_file
probes = []
for sub in ["annotations", "non_working_days", "non_representative_days"]:
    if (root / sub).exists():
        for p in first_file_per_dir(root / sub)[:12]:
            rec = probe_file(p)
            rec["path"] = os.path.relpath(rec["path"], str(root))
            probes.append(rec)
            print("=" * 100)
            print(rec["path"], "|", rec["size_bytes"], "bytes |", rec["container"], "| members:", rec["members_total"],
                  rec.get("member_extensions", ""), "| json:", rec["json_shape"], "| err:", rec["error"])
            print("members sample:", rec["members_sample"][:6])
            print("HEAD:", rec["head"][:600])

In [ ]:
# Cell 6: do the "removed images" lists refer to files that still exist? (matched by file name)
from src.data.annotation_probe import basename_index, check_listed_paths
idx = basename_index(root)
removed = []
for f in sorted((root / "removed_images").glob("*.txt")):
    r = check_listed_paths(f, idx)
    removed.append(r)
    print(r)

In [ ]:
# Cell 7: save everything for later notebooks
import json
os.makedirs(OUT, exist_ok=True)
with open(OUT + "/probe.json", "w") as f:
    json.dump({"dirs": rows, "probes": probes, "removed_lists": removed}, f, indent=2)
print("saved", OUT + "/probe.json")

## What to send back
Paste the output of cells 3 to 6. From the README and the first files we learn what `cars` and `spots` contain, whether there are vehicle boxes or masks, slot polygons and occupancy labels, and how each annotation file maps to an image. Then we choose the splits and the training data.